# AE 특화 EDA — `ae_eda_v1`  (P0 환경 + P1 EDA)

Autoencoder 개발계획 v1 **§4(P0)·§5(P1)** 실행 노트북.
목적은 두 가지 — ① 병합본 `merged_data.csv`를 **믿어도 되는가(무결성)**, ② AE 입력 설계에 필요한 **설계 파라미터 5종 확정**.
데이터 사전이 이미 답한 것(컬럼 의미·판정)은 **재검증하지 않고 인용**하고, "병합본에서도 성립하는가"만 스팟 체크한다.

### 실행 방법
1. 이 노트북을 `AE/ae_eda_v1/` 폴더에 두고 로컬 venv 커널로 **위에서 아래로 순차 실행**.
2. 최초 실행 시 `merged_data.csv → .parquet` 캐시가 자동 생성됨(수십 초). 이후엔 parquet만 읽음.
3. 마지막 **P1 DoD 요약 셀**의 출력(설계 파라미터 + 세트 경계 해시)을 그대로 공유해줘 → REPORT_01 발행.

### 불가침 규칙 (이 노트북이 코드로 강제하는 것)
- **1** C65 입력 금지 · 분할은 **C64(Wafer) 그룹** 기준 (행 단위 금지)
- **2** seg2·C6_1은 **최종 평가 전용** — §5-2에서 '관찰(read-only)'만, 설계 파라미터(§5-3~5-5)는 학습셋만 사용
- **3** 스케일러·클리핑 fit = **seg1 train만**
- **5** Step4 **과도(C42=1)/정착(C42=0) 분리는 신성불가침**
- **7** AE 입력은 **물리 신호만** (시간·카운터·설정값·C12·식별자 금지) — 채널 단일소스 = 계획서 부록 A
- **8** 노트북은 출력 비워 전달, **실행은 사용자** · **10** 계획서 vs 실측 모순 시 진행 중단→보고

> ⚠️ **§5-1 무결성 게이트에서 하나라도 FAIL이면 불가침 10에 따라 진행을 멈추고 보고 대상이다.** 자동 우회 금지.


In [1]:
# ── P0-1 · 환경·의존성 체크 (계획서 §4) ───────────────────────────────
# 목적: 이후 모든 노트북이 같은 바닥에서 돌도록 런타임을 고정·기록한다.
#       import 성공 + 버전 스냅샷 + GPU(CUDA) 인식을 REPORT 재현 근거로 남긴다.
import sys, platform, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
try:
    import IPython.display as _d
    def show(x):
        try: _d.display(x)
        except Exception: print(x.to_string() if hasattr(x,'to_string') else x)
except Exception:
    def show(x): print(x.to_string() if hasattr(x,'to_string') else x)

_env = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__}
for mod in ['pyarrow','sklearn']:
    try:
        m = __import__(mod); _env[mod] = getattr(m,'__version__','?')
    except Exception as e:
        _env[mod] = f'MISSING ({type(e).__name__})'
try:
    import torch
    _env['torch'] = torch.__version__
    _env['cuda_available'] = bool(torch.cuda.is_available())
    _env['cuda_device'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-fallback'
except Exception as e:
    _env['torch'] = f'MISSING ({type(e).__name__})'; _env['cuda_available'] = False

print('── 런타임 스냅샷 (REPORT 기록용) ──')
for k, v in _env.items():
    print(f'  {k:16s}: {v}')
assert sys.version_info >= (3, 10), 'python >= 3.10 필요 (계획서 §4)'
if not _env.get('cuda_available'):
    print('\n[주의] CUDA 미인식 — CPU 폴백 진행. 이 규모면 EDA는 CPU로 충분.')
    print('       GPU DoD(torch.cuda.is_available()==True)는 모델링 노트북(ae_v1~)에서 재확인.')
else:
    print('\n[OK] GPU 인식 — 계획서 §4 P0 GPU DoD 충족.')


── 런타임 스냅샷 (REPORT 기록용) ──
  python          : 3.12.10
  numpy           : 2.5.1
  pandas          : 2.3.3
  pyarrow         : 24.0.0
  sklearn         : 1.9.0
  torch           : MISSING (ModuleNotFoundError)
  cuda_available  : False

[주의] CUDA 미인식 — CPU 폴백 진행. 이 규모면 EDA는 CPU로 충분.
       GPU DoD(torch.cuda.is_available()==True)는 모델링 노트북(ae_v1~)에서 재확인.


In [2]:
# ── P0-2 · 재현성 시드 + 프로젝트 상수 (계획서 §4 재현성, 불가침 1·7) ──
# 목적: 시드 3종 고정 + 컬럼 그룹 상수를 단일 소스로 못박는다.
#       채널 목록의 단일 소스는 계획서 부록 A — 여기 상수는 그 사본이다.
import random
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    try:
        torch.use_deterministic_algorithms(True); _DET = 'use_deterministic_algorithms(True)'
    except Exception:
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
        _DET = 'cudnn.deterministic(fallback)'
except Exception:
    _DET = 'torch 없음 — numpy/random만 고정'
print(f'seed={SEED} 고정 · 결정성: {_DET}')

# ── 경로: 노트북은 AE/ae_eda_v1/ 에서 실행 → 원본 CSV는 부모 폴더. 견고 탐색 ──
def _find(name):
    for p in [Path(name), Path('..')/name, Path('../..')/name]:
        if p.exists(): return p.resolve()
    return (Path('..')/name).resolve()
CSV_PATH = _find('merged_data.csv')
PARQUET_PATH = CSV_PATH.with_suffix('.parquet')
print(f'CSV 경로: {CSV_PATH}')

# ── 컬럼 그룹 상수 (데이터 사전 §6 · 계획서 부록 A) ──
STR_COLS    = ['C6','C14','C20','C21','C22','C23','C24','C28','C29','C30','C38','C64']  # 접두사 문자열
CORRUPT_COL = 'C40'                                                    # 손상(날짜부 잘림) → 드랍(C10 대체)
DROP_EMPTY  = ['C2','C13','C26','C37','C43','C47','C53','C55']         # 전체 NULL 8종
DROP_DUP    = ['C36','C35','C38','C40']                                # 중복→기준키(C7/C34/C64/C10)
CONST_COLS  = ['C3','C8','C14','C19','C21','C24','C28','C29','C30','C44','C45','C51']  # 상수 12종(검증용)
GROUP_KEYS  = ['C24','C6','C7']                                        # 정규화 그룹 키(입력 아님)

# AE 입력 절대 금지 (불가침 7 · 부록 A 제외표) — 물리 신호만 재구성
FORBIDDEN_INPUT = set(
    ['C65']                                                            # 타깃
    + ['C64','C38','C20','C21','C22','C23','C34','C35','C14','C24','C6','C7','C36']  # 식별자·키
    + ['C10','C39','C40','C41','C46','C33']                            # 시간·순번·카운터(C33 포함)
    + ['C1','C4','C5','C48']                                           # 설정값(knob)
    + ['C12']                                                          # setpoint 도장
    + ['C42'])                                                         # 윈도우 플래그

# 부록 A 포함(재구성 대상) — 물리 신호
CH_SETTLE_CONT = ['C11','C17','C9','C52','C15','C16','C31','C63','C58','C57']  # 정착 연속/저변동
CH_TRANSIENT   = ['C18','C27','C32','C62','C61']                              # 점화 과도 서명
CH_DISCRETE    = ['C49','C54','C56','C50']                                    # 이산·이벤트
CH_MUX         = ['C59','C60']                                                # 멀티플렉스 재구성
SEQ_CANDIDATES = ['C11','C17','C9','C52','C15','C16','C31','C63','C32','C62','C61']  # ⓢ v2 후보

def sha1_8(items):
    # 정렬된 항목의 SHA1 앞 8자리 — 필터/세트 경계 재현 지문(재현성 규칙)
    h = hashlib.sha1()
    for x in sorted(map(str, items)):
        h.update(str(x).encode()); h.update(b'|')
    return h.hexdigest()[:8]

print(f'상수 확정: STR {len(STR_COLS)} · 전체NULL {len(DROP_EMPTY)} · 중복 {len(DROP_DUP)} · '
      f'상수 {len(CONST_COLS)} · 금지입력 {len(FORBIDDEN_INPUT)}종')


seed=42 고정 · 결정성: torch 없음 — numpy/random만 고정
CSV 경로: C:\Users\Dell3571\Documents\AE\merged_data.csv
상수 확정: STR 12 · 전체NULL 8 · 중복 4 · 상수 12 · 금지입력 26종


In [3]:
# ── P0-3 · 데이터 로딩 + parquet 캐시 (계획서 §4 로딩 표준) ────────────
# 목적: 42MB CSV 반복 파싱 제거 + dtype 고정. 최초 1회만 변환, 이후 parquet만.
#       원본 CSV는 불변(read-only) 취급.
def load_data(force_rebuild=False):
    if PARQUET_PATH.exists() and not force_rebuild:
        df = pd.read_parquet(PARQUET_PATH)
        print(f'[cache] parquet 로드: {df.shape[0]:,} × {df.shape[1]}')
        return df
    if not CSV_PATH.exists():
        raise FileNotFoundError(f'merged_data.csv 없음 — 노트북을 AE/ae_eda_v1/에 두고 실행. 탐색={CSV_PATH}')
    print('[build] CSV → parquet 최초 변환 중...')
    dtypes = {c: 'string' for c in STR_COLS}; dtypes[CORRUPT_COL] = 'string'
    df = pd.read_csv(CSV_PATH, dtype=dtypes, low_memory=False)
    df.to_parquet(PARQUET_PATH, index=False)
    print(f'[build] 저장: {PARQUET_PATH.name}  {df.shape[0]:,} × {df.shape[1]}')
    return df

df_raw = load_data()
print(f'로드 형상: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열')
print('dtype 분포:', {str(k): int(v) for k, v in df_raw.dtypes.astype(str).value_counts().items()})


[build] CSV → parquet 최초 변환 중...
[build] 저장: merged_data.parquet  164,701 × 65
로드 형상: 164,701행 × 65열
dtype 분포: {'int64': 38, 'float64': 14, 'string': 13}


In [4]:
# ── P0-4 · 표준 전처리 유틸 (계획서 §4 공통 유틸 — 이후 노트북에 동일 복제) ──
# 목적: 정렬·재구성·세그 경계·윈도우 분리를 함수로 못박아 전 노트북이 공유.
#       불가침 5(과도/정착 분리)·불가침 1(C64 그룹)을 코드로 강제.
def standard_sort(df):
    # 표준 정렬: C10↑, 동률 시 C64·C46 (계획서 §5-1). C10 3초 격자의 동률을
    # C64·C46로 결정적 정렬 → 재현성. mergesort=안정 정렬.
    return df.sort_values(['C10','C64','C46'], kind='mergesort').reset_index(drop=True)

def reconstruct_c59_c60(df, thr=1000):
    # C59/C60 멀티플렉스 → active_value + 채널 플래그 (데이터 사전 노트 3).
    # 상호배타(한 행 둘 다 >thr = 0%) 전제 — 검증은 §5-3.
    out = df.copy()
    c59_big = out['C59'] > thr
    out['C5960_active'] = np.where(c59_big, out['C59'], out['C60']).astype(float)
    out['C5960_ch']     = np.where(c59_big, 0, 1).astype('int8')   # 0=C59측 / 1=C60측
    return out

def is_step4_transient(df):   # 점화 과도 (불가침 5) — 절대 정착과 섞지 않음
    return (df['C7'] == 4) & (df['C42'] == 1)
def is_step4_settle(df):      # 정착 (불가침 5)
    return (df['C7'] == 4) & (df['C42'] == 0)

def find_regime_boundary(df):
    # C33(RF time) 리셋으로 seg1/seg2 경계 확정 (계획서 §5-2).
    # wafer 대표 C33 = 그 wafer 첫 행. 시간순으로 훑어 대표값이 급락(리셋)하는
    # 첫 wafer가 seg2 시작. 반환: 경계 정보 dict.
    w = (df.groupby('C64', sort=False)
           .agg(c33_first=('C33','first'), t_min=('C10','min'))
           .reset_index().sort_values('t_min', kind='mergesort').reset_index(drop=True))
    drop = (w['c33_first'].diff() <= -2)          # 직전 대표 대비 2↓ = 리셋
    reset_idx = np.where(drop.values)[0]
    info = {'n_reset': int(len(reset_idx)), '_wafer_order': w}
    if len(reset_idx):
        b = int(reset_idx[0])
        info.update(boundary_wafer_pos=b,
                    seg1_last_wafer=str(w.loc[b-1,'C64']), seg2_first_wafer=str(w.loc[b,'C64']),
                    boundary_time=str(pd.to_datetime(w.loc[b,'t_min'], unit='s')),
                    seg1_wafers=w.iloc[:b]['C64'].tolist(), seg2_wafers=w.iloc[b:]['C64'].tolist())
    return info

print('유틸 정의: standard_sort · reconstruct_c59_c60 · is_step4_transient/settle · find_regime_boundary')


유틸 정의: standard_sort · reconstruct_c59_c60 · is_step4_transient/settle · find_regime_boundary


## §5-1 · 병합본 무결성 게이트  *(모든 후속 작업의 전제 — 게이트)*

기대값(계획서·데이터 사전) 대 병합본 실측을 **대조표**로 판정한다.
하나라도 `★FAIL`이면 **불가침 10**: 진행 중단 → FAIL 항목 보고 → 사용자 결정. **자동 우회 금지.**


In [5]:
# ── §5-1 · 병합본 무결성 게이트 (계획서 §5-1 대조표) ──────────────────
df = standard_sort(df_raw)     # 표준 정렬 1회 (이후 df 기준)
gate = []
def chk(name, expected, actual, ok):
    gate.append({'검사': name, '기대(계획서)': str(expected), '실측': str(actual),
                 '판정': 'PASS' if ok else '★FAIL'})

# 1) 형상
chk('형상 행수', 164701, len(df), len(df) == 164701)
chk('형상 열수', 65, df.shape[1], df.shape[1] == 65)
# 2) 컬럼명 C1~C65
exp_cols = [f'C{i}' for i in range(1, 66)]
chk('컬럼명 C1~C65', 'C1..C65', 'match' if list(df.columns)==exp_cols else 'MISMATCH',
    list(df.columns) == exp_cols)
# 3) C64 nunique (wafer)
nwaf = int(df['C64'].nunique())
chk('C64 nunique(wafer)', '~15919 (F1)', nwaf, 15000 <= nwaf <= 16500)
# 4) (C64,C7,C46) 유일성
dup = int(df.duplicated(['C64','C7','C46']).sum())
chk('(C64,C7,C46) 유일성', 0, dup, dup == 0)
# 5) 기간
t0 = pd.to_datetime(df['C10'].min(), unit='s'); t1 = pd.to_datetime(df['C10'].max(), unit='s')
span = (t1 - t0).days
chk('기간(일수)', '~69일 (12/01~02/08)', f'{t0.date()}~{t1.date()} ({span}일)', 60 <= span <= 75)
# 6) 행/wafer 분포
sz = df.groupby('C64').size()
oob = sz[(sz < 9) | (sz > 24)]
chk('행/wafer 9~24', '전 wafer', f'{sz.min()}~{sz.max()} (범위밖 {len(oob)}장)', True)  # 관찰
# 7) C40 손상 → 드랍 확정
c40_iso = df['C40'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}').mean()
chk('C40 정상 날짜형식 비율', '≈0% (손상)→드랍', f'{c40_iso:.1%}', True)  # 손상 확인=드랍 근거
# 8) 전체 NULL 8종
empty = {c: int(df[c].notna().sum()) for c in DROP_EMPTY}
chk('전체NULL 8종(사전§6)', '전부 0 non-null', empty, all(v == 0 for v in empty.values()))
# 9) 상수 12종
const = {c: int(df[c].nunique(dropna=True)) for c in CONST_COLS}
chk('상수 12종(사전§6)', '전부 nunique<=1', const, all(v <= 1 for v in const.values()))
# 10) 관계식 스팟 체크
chk('C41 = C10-C39', '100%', f"{(df['C41']==(df['C10']-df['C39'])).mean():.1%}",
    (df['C41']==(df['C10']-df['C39'])).mean() > 0.999)
chk('C7 = C36', '100%', f"{(df['C7']==df['C36']).mean():.1%}", (df['C7']==df['C36']).mean() > 0.999)
chk('C34 = C35', '100%', f"{(df['C34']==df['C35']).mean():.1%}", (df['C34']==df['C35']).mean() > 0.999)
m1 = int(df.groupby('C64')['C38'].nunique().max()); m2 = int(df.groupby('C38')['C64'].nunique().max())
chk('C64 ↔ C38 1:1', 'both max=1', f'{m1}/{m2}', m1 == 1 and m2 == 1)
s4 = df[df['C7'] == 4]
c42_rule = ((s4['C42'] == 1) == (s4['C46'] <= 3)).mean()
chk('C42 = step4 C46<=3', '100%', f'{c42_rule:.1%}', c42_rule > 0.99)
# 11) 원본 정렬 상태 (참고)
chk('원본 C10 정렬', '(확인용)', f'monotonic={df_raw["C10"].is_monotonic_increasing} → 표준정렬 적용', True)
# 12) C46 순번 점프 (벡터화)
d = df.sort_values(['C64','C7','C46'], kind='mergesort')
gid = d['C64'].astype(str) + '_' + d['C7'].astype(str)
same = (gid == gid.shift())
jr = float(((d['C46'].diff() != 1) & same).sum() / same.sum())
chk('C46 순번 점프율', '~0.9%', f'{jr:.2%}', True)  # 관찰 — 완결성 피처 재료

gate_df = pd.DataFrame(gate)
GATE_PASS = bool((gate_df['판정'] == 'PASS').all())
show(gate_df)
print('\n' + ('✅ 무결성 게이트 전체 PASS — 후속 진행 가능'
              if GATE_PASS else '★ FAIL 존재 — 불가침 10: 진행 중단하고 아래를 사용자에게 보고'))
if not GATE_PASS:
    show(gate_df[gate_df['판정'] != 'PASS'])


,검사,기대(계획서),실측,판정
0,형상 행수,164701,164701,PASS
1,형상 열수,65,65,PASS
2,컬럼명 C1~C65,C1..C65,match,PASS
3,C64 nunique(wafer),~15919 (F1),15919,PASS
4,"(C64,C7,C46) 유일성",0,1456,★FAIL
5,기간(일수),~69일 (12/01~02/08),2018-11-30~2019-02-08 (69일),PASS
6,행/wafer 9~24,전 wafer,9~24 (범위밖 0장),PASS
7,C40 정상 날짜형식 비율,≈0% (손상)→드랍,0.0%,PASS
8,전체NULL 8종(사전§6),전부 0 non-null,"{'C2': 0, 'C13': 0, 'C26': 0, 'C37': 0, 'C43':...",PASS
9,상수 12종(사전§6),전부 nunique<=1,"{'C3': 1, 'C8': 1, 'C14': 1, 'C19': 1, 'C21': ...",PASS



★ FAIL 존재 — 불가침 10: 진행 중단하고 아래를 사용자에게 보고


,검사,기대(계획서),실측,판정
4,"(C64,C7,C46) 유일성",0,1456,★FAIL
10,C41 = C10-C39,100%,95.0%,★FAIL


## §5-2 · 레짐·사이클 구조  *(평가 설계의 뼈대)*

12/23 대PM(**C33 리셋**) 경계를 분할 상수로 확정하고, "AE가 잡아야 할 레짐 신호가 데이터에 실재하는가"를 스팟 확인한다.

> ⚠️ **불가침 2**: seg2·C6_1은 최종 평가 전용. 이 셀은 **관찰(read-only)** 이며, 여기서 나온 seg2/C6_1 통계는 이후 설계 셀(§5-3~5-5)에서 **참조하지 않는다**. `seg` 컬럼은 학습셋(seg1)을 **골라내는 필터**로만 쓴다.


In [6]:
# ── §5-2 · 레짐·사이클 구조 (계획서 §5-2) — 관찰 전용 ─────────────────
reg = find_regime_boundary(df)
print(f"C33 리셋(경계) 감지: {reg['n_reset']}회  (기대 1회, 12/23)")
if reg['n_reset'] >= 1:
    print(f"  seg1 마지막 wafer : {reg['seg1_last_wafer']}")
    print(f"  seg2 첫 wafer     : {reg['seg2_first_wafer']}")
    print(f"  경계 시각         : {reg['boundary_time']}")
    print(f"  seg1 = {len(reg['seg1_wafers']):,} wafer  /  seg2 = {len(reg['seg2_wafers']):,} wafer")

SEG1 = set(reg.get('seg1_wafers', [])); SEG2 = set(reg.get('seg2_wafers', []))
df['seg'] = np.where(df['C64'].isin(SEG1), 'seg1', 'seg2')  # 필터용 파생(입력 아님)

# C6_1 실험 구간 (백테스트 앵커 #2) — 관찰
c6_1_wafers = df.loc[df['C6'] == 'C6_1', 'C64'].unique().tolist()
if c6_1_wafers:
    t = df.loc[df['C6'] == 'C6_1', 'C10']
    print(f"\nC6_1 실험: {len(c6_1_wafers)}장  "
          f"({pd.to_datetime(t.min(), unit='s')} ~ {pd.to_datetime(t.max(), unit='s')})  — 학습 제외, 앵커로만")

# 레짐 서명 재현 (정착 기준) — 관찰: 신호가 데이터에 실재하는가
set_m = is_step4_settle(df)
def sig(col):
    a = pd.to_numeric(df.loc[set_m & (df['seg']=='seg1'), col], errors='coerce')
    b = pd.to_numeric(df.loc[set_m & (df['seg']=='seg2'), col], errors='coerce')
    return (b.mean() - a.mean()) / (a.std() + 1e-9)
print('\n레짐 서명 (정착, seg2−seg1, σ) — 기대: C17 큰 음(≈−4.5σ), C12 점프(F10):')
for c in ['C17','C12','C11']:
    print(f'  {c:>4}: {sig(c):+.2f}σ')
wm = df[set_m].groupby('C64').agg(c11min=('C11','min'), seg=('seg','first'))
a, b = wm[wm.seg=='seg1']['c11min'], wm[wm.seg=='seg2']['c11min']
print(f"  C11_wf_min: {(b.mean()-a.mean())/(a.std()+1e-9):+.2f}σ  (F14 기대 +6.56σ)")

# seg1 내부 정상 지형 (C23 캠페인·C22) — '정상' 분산 폭 이해용 (입력 금지)
print(f"\nseg1 내부: C23(캠페인) {df[df.seg=='seg1']['C23'].nunique()}종 · "
      f"C22(sub-lot) {df[df.seg=='seg1']['C22'].nunique()}종 — 정상 분산 폭(리스크3, C23은 입력 금지)")

# 백테스트 이벤트 시점표
events = pd.DataFrame([
    {'이벤트': '12/23 대PM(요란)', '경계/규모': reg.get('seg2_first_wafer','?'), '시각/구간': reg.get('boundary_time','?')},
    {'이벤트': 'C6_1 실험(앵커#2)', '경계/규모': f'{len(c6_1_wafers)}장', '시각/구간': '12/31~1/1'},
])
print('\n── 백테스트 이벤트 시점표 ──'); show(events)


C33 리셋(경계) 감지: 1회  (기대 1회, 12/23)
  seg1 마지막 wafer : C64_9177
  seg2 첫 wafer     : C64_9664
  경계 시각         : 2018-12-23 16:31:23
  seg1 = 4,903 wafer  /  seg2 = 11,016 wafer

C6_1 실험: 121장  (2018-12-31 03:59:54 ~ 2019-01-01 00:39:12)  — 학습 제외, 앵커로만

레짐 서명 (정착, seg2−seg1, σ) — 기대: C17 큰 음(≈−4.5σ), C12 점프(F10):
   C17: -3.22σ
   C12: +4.17σ
   C11: +0.09σ
  C11_wf_min: +0.21σ  (F14 기대 +6.56σ)

seg1 내부: C23(캠페인) 16종 · C22(sub-lot) 692종 — 정상 분산 폭(리스크3, C23은 입력 금지)

── 백테스트 이벤트 시점표 ──


,이벤트,경계/규모,시각/구간
0,12/23 대PM(요란),C64_9664,2018-12-23 16:31:23
1,C6_1 실험(앵커#2),121장,12/31~1/1


## §5-3 · 입력 채널 선정 EDA

부록 A 채널을 실데이터로 확정한다. 통계는 **학습셋(seg1 & C6_0 & Step4)** 에서만 — 불가침 2·3.
과도(C42=1)/정착(C42=0)은 **분리 집계**(불가침 5), C59/C60은 재구성 검증.


In [7]:
# ── §5-3 · 입력 채널 선정 EDA (계획서 §5-3 · 부록 A 확정) ──────────────
train_mask = (df['seg'] == 'seg1') & (df['C6'] == 'C6_0')      # 학습셋 정의 (불가침 2·4)
df_tr = reconstruct_c59_c60(df[train_mask].copy())
tr_set = df_tr[is_step4_settle(df_tr)]     # 정착
tr_tra = df_tr[is_step4_transient(df_tr)]  # 과도
print(f'학습셋(seg1·C6_0): {df_tr["C64"].nunique():,} wafer / 정착 {len(tr_set):,}행 · 과도 {len(tr_tra):,}행')

# (a) C59/C60 재구성 검증 — 상호배타
both = float(((df_tr['C59'] > 1000) & (df_tr['C60'] > 1000)).mean())
print(f'\n(a) C59/C60 상호배타: 둘 다 >1000 인 행 = {both:.2%}  (기대 0%)')

# (b) 정착 연속 채널: 분포·왜도(로그 필요성)·wafer내 std=0 비율(설정값 잔존 반전 신호)
rows = []
for c in CH_SETTLE_CONT + ['C5960_active']:
    if c not in tr_set.columns: continue
    s = pd.to_numeric(tr_set[c], errors='coerce').dropna()
    if not len(s): continue
    zstd = tr_set.groupby('C64')[c].std()
    rows.append({'채널': c, 'mean': round(s.mean(),2), 'std': round(s.std(),2),
                 'min': round(s.min(),2), 'max': round(s.max(),2), '왜도': round(s.skew(),2),
                 'wafer내std0비율': f'{float((zstd.fillna(0)==0).mean()):.0%}',
                 'log검토': 'Y' if abs(s.skew()) >= 3 else ''})
print('\n(b) [정착 연속 채널 분포 — 학습셋]'); show(pd.DataFrame(rows))

# (c) 점화 과도 서명 채널 — 과도 구간에서만 (불가침 5)
rows2 = []
for c in CH_TRANSIENT:
    if c not in tr_tra.columns: continue
    s = pd.to_numeric(tr_tra[c], errors='coerce').dropna()
    if len(s): rows2.append({'채널': c, '과도mean': round(s.mean(),2),
                             '과도max': round(s.max(),2), '과도min': round(s.min(),2)})
print('\n(c) [점화 과도 서명 — 정착과 분리 집계]'); show(pd.DataFrame(rows2))

# (d) 이산 채널 처리안 (연속취급+가중하향 vs 제외)
print('\n(d) [이산 채널 점검]')
for c in CH_DISCRETE + ['C57','C32']:
    if c in df_tr.columns:
        nun = int(df_tr[c].nunique())
        print(f'  {c:>4}: nunique={nun:>5}  ({"이산" if nun <= 25 else "연속"} 후보)')
print('→ v1 기본: 연속 취급 + 손실 가중 하향(부록 A). 최종은 학습곡선(P4)에서.')


학습셋(seg1·C6_0): 4,903 wafer / 정착 8,438행 · 과도 15,045행

(a) C59/C60 상호배타: 둘 다 >1000 인 행 = 0.00%  (기대 0%)

(b) [정착 연속 채널 분포 — 학습셋]


,채널,mean,std,min,max,왜도,wafer내std0비율,log검토
0,C11,-228.97,94.39,-351.00,-2.00,0.61,34%,
1,C17,331.29,25.94,240.00,353.00,-1.83,35%,
2,C9,64.30,1.51,61.73,69.23,1.43,34%,
3,C52,17.83,1.88,6.00,33.00,1.13,80%,
4,C15,69.48,67.41,0.00,164.00,0.68,34%,
5,C16,740.28,624.62,235.00,1588.00,0.55,34%,
6,C31,166.43,90.93,3.00,301.00,-0.06,34%,
7,C63,237.63,11.06,200.00,260.00,-0.84,34%,
8,C58,11.98,0.09,11.55,12.43,0.38,42%,
9,C57,11.47,1.02,10.00,17.00,3.09,53%,Y



(c) [점화 과도 서명 — 정착과 분리 집계]


,채널,과도mean,과도max,과도min
0,C18,1.89,20.71,0.0
1,C27,9.17,347.00,-320.0
2,C32,56.85,292.00,0.0
3,C62,10.75,1850.00,-3706.0
4,C61,-238.62,1939.00,-1160.0



(d) [이산 채널 점검]
   C49: nunique=    3  (이산 후보)
   C54: nunique=   19  (이산 후보)
   C56: nunique=   21  (이산 후보)
   C50: nunique=    3  (이산 후보)
   C57: nunique=   67  (연속 후보)
   C32: nunique=  127  (연속 후보)
→ v1 기본: 연속 취급 + 손실 가중 하향(부록 A). 최종은 학습곡선(P4)에서.


## §5-4 · 시퀀스 구조 EDA  *(v2 설계 근거 — 분기 판정)*

Step4 **정착 길이 분포**로 v2 고정 길이 `T`와 **분기 조건**을 결정한다.
**분기 규칙(계획서 §5-4)**: 정착 `p50 < 8행` 이면 Conv1D 대신 **flatten-MLP**로 전환.


In [8]:
# ── §5-4 · 시퀀스 구조 EDA (계획서 §5-4 — v2 분기 판정) ───────────────
# step별 wafer당 행수 (학습셋)
sr = df_tr.groupby(['C64','C7']).size().groupby('C7').agg(['mean','min','max']).round(1)
print('[step별 wafer당 행수 — 학습셋]'); show(sr)

# Step4 정착 길이 분위수 → T·분기 판정
sl = df_tr[is_step4_settle(df_tr)].groupby('C64').size()
q = sl.quantile([.05, .5, .9, .95]).round(1)
p50, p90 = float(q[.5]), float(q[.9])
print(f'\nStep4 정착 길이(학습셋, wafer당 행수): '
      f'p5={q[.05]} · p50={p50} · p90={p90} · p95={q[.95]} · min={sl.min()} max={sl.max()}')
BRANCH = 'flatten-MLP (Conv 부적합)' if p50 < 8 else 'Conv1D 가능'
T_FIXED = int(p90)
print(f'  ▶ 분기 판정: 정착 p50={p50} {"<" if p50 < 8 else ">="} 8  →  v2 = {BRANCH}')
print(f'  ▶ v2 고정 길이 T = p90 = {T_FIXED} (초과 절단 · 미만 패딩+마스크)')

# 레시피별 길이차 (C6_1 식각 2배 기대) — 전체 관찰(C6_1 참조만)
lbr = df[is_step4_settle(df)].groupby(['C6','C64']).size().groupby('C6').median()
print(f'\n레시피별 정착 길이 중앙값(관찰): {dict(lbr.round(1))}  — C6_1이 긺(≈2배) 기대')

# 스텝 누락 wafer (<5 step)
spw = df_tr.groupby('C64')['C7'].nunique()
miss = (spw < 5)
print(f'\n스텝 누락(<5 step) wafer(학습셋): {int(miss.sum())}장 ({float(miss.mean()):.1%}) '
      f'— 완결성 피처 재료 + 스코어링 가능 여부(§6-3 최소치)')


[step별 wafer당 행수 — 학습셋]


,mean,min,max
C7,,,
1,2.1,1,8
4,4.8,4,12
5,1.0,1,2
6,1.4,1,4
7,1.7,1,4



Step4 정착 길이(학습셋, wafer당 행수): p5=1.0 · p50=2.0 · p90=2.0 · p95=2.0 · min=1 max=6
  ▶ 분기 판정: 정착 p50=2.0 < 8  →  v2 = flatten-MLP (Conv 부적합)
  ▶ v2 고정 길이 T = p90 = 2 (초과 절단 · 미만 패딩+마스크)

레시피별 정착 길이 중앙값(관찰): {'C6_0': np.float64(2.0), 'C6_1': np.float64(4.0)}  — C6_1이 긺(≈2배) 기대

스텝 누락(<5 step) wafer(학습셋): 3240장 (66.1%) — 완결성 피처 재료 + 스코어링 가능 여부(§6-3 최소치)


## §5-5 · 정규화 전략 EDA

정규화 그룹 축 확인 + **Robust(median/IQR)** 경계 + **winsorize(P0.1/P99.9)** 클립 경계 산출.
**fit 범위 = 학습셋(seg1 TRAIN)만** — 불가침 3. seg2 이동이 정규화로 씻겨나가면 레짐 감지가 불능이 된다.
여기서는 **경계값 산출만** 하고, 실제 scaler fit은 P4에서.


In [9]:
# ── §5-5 · 정규화 전략 EDA (계획서 §5-5 · 불가침 3) ───────────────────
print('[정규화 그룹 축] Chamber(C24) × Recipe(C6) × Step(C7) × Window(C42)')
print(f'  C24: {df_tr["C24"].nunique()}종(train 단일 — 챔버 축은 인터페이스만 개방)'
      f' · C6: 학습셋 C6_0 단일 · C7: {sorted(df_tr["C7"].unique().tolist())}')

# 학습셋 내부 시간순 TRAIN/VAL 80/20 (VAL=seg1 후반 20%, C64 그룹 — 계획서 §6-3)
order = df_tr.groupby('C64')['C10'].min().sort_values()
cut = int(len(order) * 0.8)
TRAIN_W = set(order.index[:cut]); VAL_W = set(order.index[cut:])
print(f'\nTRAIN {len(TRAIN_W):,} wafer / VAL {len(VAL_W):,} wafer (seg1 시간순 80/20, C64 그룹)')

# winsorize(P0.1/P99.9) + Robust(median/IQR) — TRAIN 정착만으로 산출 (불가침 3)
fit_set = df_tr[is_step4_settle(df_tr) & df_tr['C64'].isin(TRAIN_W)]
rows = []
for c in CH_SETTLE_CONT + ['C5960_active']:
    if c not in fit_set.columns: continue
    s = pd.to_numeric(fit_set[c], errors='coerce').dropna()
    if not len(s): continue
    q1, q3 = s.quantile(.25), s.quantile(.75)
    rows.append({'채널': c, 'median': round(s.median(),2), 'IQR': round(q3-q1,3),
                 'clip_lo(P0.1)': round(s.quantile(.001),2), 'clip_hi(P99.9)': round(s.quantile(.999),2),
                 '스케일러': 'Robust' if abs(s.skew()) < 3 else 'Robust+log'})
norm_tbl = pd.DataFrame(rows)
print('\n[채널별 정규화 경계 — TRAIN fit, 불가침 3]'); show(norm_tbl)
print('\n주: 이 경계는 산출값. 실제 scaler는 P4에서 이 규칙으로 fit. 재fit은 CT②(§10-4)에서만.')


[정규화 그룹 축] Chamber(C24) × Recipe(C6) × Step(C7) × Window(C42)
  C24: 1종(train 단일 — 챔버 축은 인터페이스만 개방) · C6: 학습셋 C6_0 단일 · C7: [1, 4, 5, 6, 7]

TRAIN 3,922 wafer / VAL 981 wafer (seg1 시간순 80/20, C64 그룹)

[채널별 정규화 경계 — TRAIN fit, 불가침 3]


,채널,median,IQR,clip_lo(P0.1),clip_hi(P99.9),스케일러
0,C11,-247.00,147.000,-350.00,-2.00,Robust
1,C17,343.00,15.000,293.00,352.00,Robust
2,C9,63.73,1.749,61.86,69.17,Robust
3,C52,18.00,1.000,6.00,32.00,Robust
4,C15,23.00,141.000,0.00,164.00,Robust
5,C16,258.00,1341.000,235.00,1588.00,Robust
6,C31,167.00,162.000,4.00,300.00,Robust
7,C63,240.00,15.000,205.00,257.25,Robust
8,C58,11.98,0.010,11.57,12.41,Robust
9,C57,11.00,1.000,11.00,17.00,Robust+log



주: 이 경계는 산출값. 실제 scaler는 P4에서 이 규칙으로 fit. 재fit은 CT②(§10-4)에서만.


## P1 DoD — 설계 파라미터 5종 자동 요약

아래 5종이 숫자로 확정되면 P1 완료. **세트 경계 해시**는 필터 재현 지문 — REPORT_01·README에 상수로 고정한다.
> 이 셀의 출력 전체를 그대로 공유해줘 → REPORT_01(수령 검증표) 발행.


In [10]:
# ── P1 DoD 자동 요약 (계획서 §5 P1 DoD 5종) ───────────────────────────
print('=' * 66); print('  P1 DoD — 설계 파라미터 확정 요약  (ae_eda_v1)'); print('=' * 66)

seg1_h, seg2_h = sha1_8(SEG1), sha1_8(SEG2)
train_h, val_h = sha1_8(TRAIN_W), sha1_8(VAL_W)
c6_1_h = sha1_8(c6_1_wafers)

print(f'\n[DoD1 · 전처리 규칙 확정]')
print(f'   정렬=C10↑·C64·C46 | 드랍=전체NULL{len(DROP_EMPTY)}+중복{len(DROP_DUP)}(C40손상 포함)')
print(f'   C59/60→active_value+플래그 재구성 | 과도/정착 분리(불가침5)')
print(f'\n[DoD2 · 세트 경계 상수 + 해시]')
print(f'   seg1={len(SEG1):,}[{seg1_h}]  seg2={len(SEG2):,}[{seg2_h}]  (경계={reg.get("seg2_first_wafer","?")}, {reg.get("boundary_time","?")})')
print(f'   TRAIN={len(TRAIN_W):,}[{train_h}]  VAL={len(VAL_W):,}[{val_h}]  C6_1={len(c6_1_wafers)}[{c6_1_h}]')
print(f'\n[DoD3 · 입력 채널 목록 (부록 A)]')
print(f'   정착연속 {len(CH_SETTLE_CONT)} + 과도 {len(CH_TRANSIENT)} + 이산 {len(CH_DISCRETE)} + MUX재구성(C59/60)')
print(f'   금지입력 {len(FORBIDDEN_INPUT)}종 격리(C65·C33·C12·knob·식별자) — 불가침 7')
print(f'\n[DoD4 · v2 시퀀스 길이·분기]')
print(f'   정착 p50={p50} · T(p90)={T_FIXED} · 분기={BRANCH}')
print(f'\n[DoD5 · 무결성 게이트]')
print(f'   {"PASS ✅" if GATE_PASS else "★FAIL — 불가침10 진행중단"}  (§5-1 대조표 {int((gate_df["판정"]=="PASS").sum())}/{len(gate_df)} PASS)')

print('\n' + '─' * 66)
print('세트 경계 해시 (재현 지문 — REPORT/README에 고정):')
print(f'  seg1:{seg1_h}  seg2:{seg2_h}  train:{train_h}  val:{val_h}  c6_1:{c6_1_h}')
print(f'  seed={SEED}  |  parquet={PARQUET_PATH.name}')
print('─' * 66)
print('\n다음: 이 출력을 공유 → "컴파일 완료 됐어!" → REPORT_01 발행 + P2(피처명세)/P3(하네스) 착수 제안')


  P1 DoD — 설계 파라미터 확정 요약  (ae_eda_v1)

[DoD1 · 전처리 규칙 확정]
   정렬=C10↑·C64·C46 | 드랍=전체NULL8+중복4(C40손상 포함)
   C59/60→active_value+플래그 재구성 | 과도/정착 분리(불가침5)

[DoD2 · 세트 경계 상수 + 해시]
   seg1=4,903[1b142963]  seg2=11,016[0a01fcf4]  (경계=C64_9664, 2018-12-23 16:31:23)
   TRAIN=3,922[6ac80376]  VAL=981[fa500f3d]  C6_1=121[160c72e8]

[DoD3 · 입력 채널 목록 (부록 A)]
   정착연속 10 + 과도 5 + 이산 4 + MUX재구성(C59/60)
   금지입력 26종 격리(C65·C33·C12·knob·식별자) — 불가침 7

[DoD4 · v2 시퀀스 길이·분기]
   정착 p50=2.0 · T(p90)=2 · 분기=flatten-MLP (Conv 부적합)

[DoD5 · 무결성 게이트]
   ★FAIL — 불가침10 진행중단  (§5-1 대조표 15/17 PASS)

──────────────────────────────────────────────────────────────────
세트 경계 해시 (재현 지문 — REPORT/README에 고정):
  seg1:1b142963  seg2:0a01fcf4  train:6ac80376  val:fa500f3d  c6_1:160c72e8
  seed=42  |  parquet=merged_data.parquet
──────────────────────────────────────────────────────────────────

다음: 이 출력을 공유 → "컴파일 완료 됐어!" → REPORT_01 발행 + P2(피처명세)/P3(하네스) 착수 제안


---
### 이 노트북이 확정하는 것 (요약)

| P1 DoD | 산출 셀 | 내용 |
|---|---|---|
| 1 전처리 규칙 | P0-4 · §5-1 | 정렬·드랍·C59/60 재구성·윈도우 분리 |
| 2 세트 경계 | §5-2 · §5-5 | seg1/seg2/C6_1/TRAIN/VAL + 해시 |
| 3 입력 채널 | §5-3 | 부록 A 채널 확정 + 금지입력 격리 |
| 4 v2 길이·분기 | §5-4 | T=p90, Conv1D vs flatten-MLP |
| 5 REPORT 재료 | §5-1 · DoD셀 | 수령 검증 대조표 + 파라미터 |

**정직한 한계 (REPORT에 명시 예정)**: ① seg2·C6_1은 관찰만 했고 설계에 불참(누수 차단) — 그래서 이 노트북은 "정상(seg1)의 지형"만 확정한다. ② 표본 경계(대PM 1회·C6_1 1구간)는 세계관 한계 그대로. ③ 무결성 게이트가 FAIL이면 이 아래 결론은 전부 보류.

다음 단계(P2/P3)는 이 노트북의 확정값을 **입력**으로 받는다 — 여기서 홀드아웃을 건드리지 않았기 때문에 이후 평가가 정직할 수 있다.
